In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

DATA_DIR = Path('dataset')
assert DATA_DIR.exists(), f"Missing {DATA_DIR.resolve()}"

# Note: in this repo, use the project virtualenv interpreter in `myenv/`
# if your default `python` cannot import pandas without crashing.

## Stripe Subscriptions candidate analysis

This notebook:
- Loads all merchant/payment files under `dataset/`
- Standardizes column names
- Builds merchant-level features to detect recurring payment behavior
- Produces an interpretable candidate score and ranked output tables

In [2]:
def _snake(s: str) -> str:
    return (
        str(s)
        .strip()
        .lower()
        .replace(' ', '_')
        .replace('-', '_')
        .replace('__', '_')
    )


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_snake(c) for c in df.columns]

    # common aliases
    rename = {
        'merchant_id': 'merchant',
        'merchantid': 'merchant',
        'id_merchant': 'merchant',
        'payment_links_volume': 'payment_link_volume',
        'paymentlink_volume': 'payment_link_volume',
        'total_payment_volume': 'total_volume',
        'total_payments_volume': 'total_volume',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    return df


def load_all_tables(data_dir: Path) -> dict[str, pd.DataFrame]:
    """Load all xlsx/csv from dataset/ and return by filename."""
    tables: dict[str, pd.DataFrame] = {}

    for p in sorted(list(data_dir.rglob('*.xlsx')) + list(data_dir.rglob('*.csv'))):
        name = p.name
        if name.startswith('~$'):
            continue
        try:
            if p.suffix.lower() == '.xlsx':
                tables[name] = pd.read_excel(p)
            else:
                tables[name] = pd.read_csv(p)
        except Exception as e:
            print(f"Skipping {p} due to error: {e}")

    return tables


tables = load_all_tables(DATA_DIR)
list(tables.keys())

['Data descriptions.xlsx',
 'Data-descriptions.csv',
 'dstakehome_merchants.csv',
 'dstakehome_merchants.xlsx',
 'dstakehome_payments.xlsx']

In [3]:
# Identify the payments + merchants tables by the columns they contain

def pick_table_by_required_cols(tables: dict[str, pd.DataFrame], required: set[str]) -> tuple[str, pd.DataFrame]:
    candidates = []
    for name, df in tables.items():
        sdf = standardize_columns(df)
        cols = set(sdf.columns)
        if required.issubset(cols):
            candidates.append((name, sdf))
    if not candidates:
        raise ValueError(f"No table found with required columns: {sorted(required)}")
    # prefer xlsx over csv if duplicates exist
    candidates.sort(key=lambda x: (0 if x[0].endswith('.xlsx') else 1, x[0]))
    return candidates[0]

payments_name, payments_df = pick_table_by_required_cols(
    tables,
    required={'date', 'merchant'}
)

# heuristic: merchants table likely has industry/business_size and merchant id
merchants_name = None
merchants_df = None
for name, df in tables.items():
    sdf = standardize_columns(df)
    if 'merchant' in sdf.columns and any(c in sdf.columns for c in ['industry', 'business_size', 'country', 'first_charge_date']):
        merchants_name, merchants_df = name, sdf
        break

payments_name, (merchants_name or 'None')

('dstakehome_payments.xlsx', 'dstakehome_merchants.csv')

In [4]:
payments_df = standardize_columns(payments_df)
if merchants_df is not None:
    merchants_df = standardize_columns(merchants_df)

# detect volume columns present
volume_cols = [c for c in payments_df.columns if c.endswith('_volume') or c.endswith('_vol') or c in ['total_volume']]
print('Payments columns:', payments_df.columns.tolist())
print('Detected volume cols:', volume_cols)

# Normalize expected product columns if present
expected = ['subscription_volume', 'checkout_volume', 'payment_link_volume', 'pos_volume', 'total_volume']
for c in expected:
    if c not in payments_df.columns:
        payments_df[c] = 0

# parse dates robustly
payments_df['date'] = pd.to_datetime(payments_df['date'], errors='coerce', utc=True)
payments_df = payments_df.dropna(subset=['date'])
payments_df['merchant'] = payments_df['merchant'].astype(str)

if merchants_df is not None:
    merchants_df['merchant'] = merchants_df['merchant'].astype(str)

payments_df.head()

Payments columns: ['date', 'merchant', 'subscription_volume', 'checkout_volume', 'payment_link_volume', 'total_volume']
Detected volume cols: ['subscription_volume', 'checkout_volume', 'payment_link_volume', 'total_volume']


,date,merchant,subscription_volume,checkout_volume,payment_link_volume,total_volume,pos_volume
0,2041-05-01 00:00:00+00:00,5d03e714,0,0,0,425340,0
1,2041-05-01 00:00:00+00:00,da22f154,0,0,0,17326,0
2,2041-05-01 00:00:00+00:00,687eebc8,79400,0,0,79400,0
3,2041-05-01 00:00:00+00:00,de478470,268400,0,0,268400,0
4,2041-05-01 00:00:00+00:00,1e719b8a,0,19895,0,19895,0


In [15]:
# Aggregate to daily merchant-level (sum in case multiple rows per day)

daily = (
    payments_df
    .groupby(['merchant', payments_df['date'].dt.floor('D')], as_index=False)[
        ['subscription_volume', 'checkout_volume', 'payment_link_volume', 'pos_volume', 'total_volume']
    ]
    .sum()
    .rename(columns={'date': 'day'})
)

# If total_volume not provided or inconsistent, recompute as sum of product volumes when possible
product_sum = daily[['subscription_volume', 'checkout_volume', 'payment_link_volume', 'pos_volume']].sum(axis=1)
# prefer provided total_volume if it is non-zero; otherwise use product sum
is_total_missing = (daily['total_volume'].fillna(0) == 0) & (product_sum > 0)
daily.loc[is_total_missing, 'total_volume'] = product_sum[is_total_missing]

# Ensure non-negative
for c in ['subscription_volume', 'checkout_volume', 'payment_link_volume', 'pos_volume', 'total_volume']:
    daily[c] = pd.to_numeric(daily[c], errors='coerce').fillna(0)
    daily.loc[daily[c] < 0, c] = 0

# Merge merchant attributes (optional)
if merchants_df is not None:
    keep_cols = [c for c in ['merchant', 'industry', 'business_size', 'country', 'first_charge_date'] if c in merchants_df.columns]
    attrs = merchants_df[keep_cols].drop_duplicates('merchant')
    daily = daily.merge(attrs, on='merchant', how='left')

print('Daily rows:', len(daily), 'Merchants:', daily['merchant'].nunique())
daily.head(-50)

Daily rows: 1577865 Merchants: 23620


,merchant,day,subscription_volume,checkout_volume,payment_link_volume,pos_volume,total_volume,industry,business_size,country,first_charge_date
0,0,2041-06-11 00:00:00+00:00,0,0,0,0,2440,NaN,NaN,NaN,NaN
1,0,2041-06-17 00:00:00+00:00,0,0,0,0,14434,NaN,NaN,NaN,NaN
2,0,2041-07-15 00:00:00+00:00,1003,1003,0,0,1003,NaN,NaN,NaN,NaN
3,0,2041-07-24 00:00:00+00:00,0,0,0,0,14305,NaN,NaN,NaN,NaN
4,0,2041-07-25 00:00:00+00:00,0,0,0,0,6390,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1577810,fffe09ec,2041-09-10 00:00:00+00:00,0,0,0,0,3500,Personal services,small,US,2038-04-06 00:00:00+00:00
1577811,fffe09ec,2041-09-12 00:00:00+00:00,0,0,0,0,20000,Personal services,small,US,2038-04-06 00:00:00+00:00
1577812,fffe09ec,2041-09-16 00:00:00+00:00,0,0,0,0,15000,Personal services,small,US,2038-04-06 00:00:00+00:00
1577813,fffe09ec,2041-09-17 00:00:00+00:00,0,0,0,0,20000,Personal services,small,US,2038-04-06 00:00:00+00:00


In [19]:
daily = daily.dropna()

In [30]:
def coef_var(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.nan
    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=0))
    if mu == 0:
        return np.nan
    return float(sd / mu)


def repeated_price_signal(volumes: np.ndarray, rounding: int = 1000) -> float:
    """Share of volume explained by the top-3 repeated daily amounts.

    rounding is in cents (1000 cents = $10). This approximates repeated plan price points.
    """
    v = np.asarray(volumes, dtype=float)
    v = v[v > 0]
    if len(v) < 5:
        return np.nan
    rounded = (np.round(v / rounding) * rounding).astype(int)
    df = pd.DataFrame({'amt': rounded, 'v': v})
    by = df.groupby('amt', as_index=False)['v'].sum().sort_values('v', ascending=False)
    top3 = by.head(3)['v'].sum()
    total = by['v'].sum()
    if total == 0:
        return np.nan
    return float(top3 / total)


def gap_regularity_share(days: pd.Series, target_gap: int, tol: int = 1) -> float:
    """Share of inter-payment gaps close to a target cadence (e.g. 7 or 30 days)."""
    if len(days) < 6:
        return np.nan
    d = pd.to_datetime(days).sort_values().dropna()
    gaps = d.diff().dt.days.dropna().to_numpy()
    if len(gaps) == 0:
        return np.nan
    return float(np.mean(np.abs(gaps - target_gap) <= tol))


def merchant_features(daily_df: pd.DataFrame) -> pd.DataFrame:
    """Compute merchant-level features without building a dense daily calendar per merchant."""
    out = []

    for m, g in daily_df.groupby('merchant'):
        g = g.sort_values('day')

        days = g['day']
        start, end = days.min(), days.max()
        span_days = int((end - start).days) + 1 if pd.notna(start) and pd.notna(end) else 0

        totals_active = g['total_volume'].to_numpy(dtype=float)
        active_days = int((totals_active > 0).sum())
        active_ratio = active_days / span_days if span_days else 0

        # Weekly totals (sparse, only active days contribute)
        weekly_totals = g.set_index('day')['total_volume'].resample('W').sum().to_numpy(dtype=float)
        weekly_cv = coef_var(weekly_totals[weekly_totals > 0]) if np.any(weekly_totals > 0) else np.nan

        # Trend: slope of weekly totals (normalized)
        if len(weekly_totals) >= 6 and np.any(weekly_totals > 0):
            x = np.arange(len(weekly_totals))
            y = weekly_totals
            slope = np.polyfit(x, y, 1)[0]
            trend_norm = float(slope / (np.mean(y[y > 0]) + 1e-9))
        else:
            trend_norm = np.nan

        subs = float(g['subscription_volume'].sum())
        chk = float(g['checkout_volume'].sum())
        pl = float(g['payment_link_volume'].sum())
        tot = float(g['total_volume'].sum())

        frac_subs = subs / tot if tot else 0
        frac_chk = chk / tot if tot else 0
        frac_pl = pl / tot if tot else 0


        feat = {
            'merchant': m,
            'span_days': span_days,
            'active_days': active_days,
            'active_ratio': active_ratio,
            'total_volume': tot,
            'avg_daily_volume_active_days': float(np.mean(totals_active[totals_active > 0])) if np.any(totals_active > 0) else 0,
            'cv_daily_active': coef_var(totals_active[totals_active > 0]),
            'cv_weekly': weekly_cv,
            'gap_share_7d': gap_regularity_share(days, target_gap=7, tol=1),
            'gap_share_30d': gap_regularity_share(days, target_gap=30, tol=2),
            'repeated_price_share_top3': repeated_price_signal(totals_active[totals_active > 0], rounding=1000),
            'trend_weekly_norm': trend_norm,
            'frac_subscription': frac_subs,
            'frac_checkout': frac_chk,
            'frac_payment_link': frac_pl,
        }

        # carry attributes if present
        for c in ['industry', 'business_size']:
            if c in daily_df.columns:
                val = g[c].dropna().iloc[0] if g[c].notna().any() else np.nan
                feat[c] = val

        out.append(feat)

    return pd.DataFrame(out)


mfeat = merchant_features(daily)


,merchant,span_days,active_days,active_ratio,total_volume,avg_daily_volume_active_days,cv_daily_active,cv_weekly,gap_share_7d,gap_share_30d,repeated_price_share_top3,trend_weekly_norm,frac_subscription,frac_checkout,frac_payment_link,industry,business_size
0,00085cb5,401,48,0.119701,5160000.0,107500.000000,1.281622,1.107514,0.276596,0.021277,0.287190,-0.019842,0.000000,1.000000,0.0,Software,small
1,000a45ba,385,37,0.096104,142500.0,3851.351351,0.597502,0.950871,0.138889,0.055556,0.691228,-0.008872,0.649123,0.000000,0.0,Others,small
2,000e2c0e,391,73,0.186701,987007.0,13520.643836,1.028213,1.048638,0.152778,0.000000,0.535188,-0.018546,0.526971,0.000000,0.0,Education,small
3,001111d7,60,9,0.150000,1865995.0,207332.777778,1.916113,1.580619,0.125000,0.000000,0.899640,0.299163,0.000000,0.000000,0.0,Transportation & car rental,small
4,0015c036,369,68,0.184282,16200.0,238.235294,0.330117,0.474787,0.000000,0.000000,1.000000,0.001115,1.000000,0.851852,0.0,Others,small
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23492,ffe98e3f,400,369,0.922500,63533332.0,172177.051491,1.648871,1.551133,0.000000,0.000000,0.118999,-0.005538,0.000000,0.000000,0.0,Ticketing & events,small
23493,ffee7774,283,171,0.604240,144938.0,847.590643,1.237194,0.683101,0.017647,0.000000,0.832342,0.018800,0.000000,0.000000,0.0,Software,small
23494,fff31b9b,170,23,0.135294,1550580.0,67416.521739,3.003768,2.026288,0.090909,0.000000,0.865414,-0.022290,0.000000,0.000000,0.0,Clothing & accessory,small
23495,fffe09ec,414,76,0.183575,2857500.0,37598.684211,1.039230,0.827876,0.040000,0.000000,0.438110,-0.002852,0.150516,0.000000,0.0,Personal services,small


In [16]:
# Scoring heuristic (0-100), designed to be business-interpretable.
# Idea: reward (1) enough data, (2) cadence/regularity, (3) stability, (4) repeated price points,
# and (5) "subscription opportunity" when recurring patterns exist but subscription usage is low.

def clamp01(x: float) -> float:
    if pd.isna(x):
        return 0.0
    return float(max(0.0, min(1.0, x)))


def score_row(r: pd.Series) -> tuple[float, dict[str, float], str]:
    contrib: dict[str, float] = {}

    # Data sufficiency
    suff = clamp01((min(r['active_days'], 60) / 60) * (min(r['span_days'], 120) / 120))
    contrib['data_sufficiency'] = 15 * suff

    # Regularity signals (gap-based cadence)
    weekly = clamp01((r.get('gap_share_7d', np.nan) - 0.15) / 0.55)   # ~0.15..0.70
    monthly = clamp01((r.get('gap_share_30d', np.nan) - 0.10) / 0.50)  # ~0.10..0.60
    contrib['weekly_regularity'] = 15 * weekly
    contrib['monthly_regularity'] = 10 * monthly

    # Stability / predictability (lower CV is better)
    cvd = r.get('cv_daily_active', np.nan)
    cvw = r.get('cv_weekly', np.nan)
    stable_daily = clamp01((2.0 - (cvd if not pd.isna(cvd) else 2.0)) / 2.0)  # cv 0..2
    stable_weekly = clamp01((1.5 - (cvw if not pd.isna(cvw) else 1.5)) / 1.5)  # cv 0..1.5
    contrib['stability_daily'] = 15 * stable_daily
    contrib['stability_weekly'] = 10 * stable_weekly

    # Repeated price points (proxy for plan pricing)
    rep = r.get('repeated_price_share_top3', np.nan)
    rep_score = clamp01((rep - 0.25) / 0.55)  # 0.25..0.80
    contrib['repeated_price_points'] = 10 * rep_score

    # Opportunity: strong recurring signals but low subscription fraction
    recurring_strength = clamp01(0.65 * weekly + 0.35 * stable_daily)
    low_sub = clamp01((0.25 - r.get('frac_subscription', 0.0)) / 0.25)  # 0..0.25
    contrib['subscription_opportunity'] = 15 * (recurring_strength * low_sub)

    # Size sanity: avoid tiny merchants dominating by perfect regularity
    tot = r.get('total_volume', 0.0)
    size = clamp01(np.log10(1 + tot) / np.log10(1 + 5e6))  # 5e6 cents = $50k
    contrib['volume_scale'] = 10 * size

    # Industry prior (small, only if available)
    industry = str(r.get('industry', '')).lower()
    industry_boost = 0.0
    if industry:
        if any(k in industry for k in ['software', 'education', 'membership', 'religion', 'fitness', 'health', 'business services']):
            industry_boost = 1.0
        elif any(k in industry for k in ['retail', 'restaurant', 'food', 'travel', 'events']):
            industry_boost = 0.3
    contrib['industry_fit'] = 5 * industry_boost

    score = float(sum(contrib.values()))

    # Human-readable pattern label
    if weekly >= 0.6:
        pattern = 'Strong weekly cadence (many ~7-day gaps)'
    elif monthly >= 0.6:
        pattern = 'Strong monthly cadence (many ~30-day gaps)'
    elif rep_score >= 0.7:
        pattern = 'Repeated price-point behavior (plan-like amounts)'
    elif stable_daily >= 0.7 and r.get('active_ratio', 0) >= 0.5:
        pattern = 'Consistent payments on many days'
    else:
        pattern = 'Some repeat activity, weaker cadence signals'

    return score, contrib, pattern


scores = []
for _, r in mfeat.iterrows():
    s, contrib, pattern = score_row(r)
    scores.append((s, contrib, pattern))

mfeat = mfeat.copy()
mfeat['candidate_score'] = [s for s, _, _ in scores]
mfeat['pattern_detected'] = [p for _, _, p in scores]
mfeat['score_breakdown'] = [c for _, c, _ in scores]

mfeat.sort_values('candidate_score', ascending=False).head(10)

,merchant,span_days,active_days,active_ratio,total_volume,avg_daily_volume_active_days,cv_daily_active,cv_weekly,gap_share_7d,gap_share_30d,repeated_price_share_top3,trend_weekly_norm,frac_subscription,frac_checkout,frac_payment_link,frac_pos,industry,business_size,candidate_score,pattern_detected,score_breakdown
3368,23ff5133,418,76,0.181818,5347126.0,70356.921053,0.161254,0.377238,0.640000,0.0,0.849504,0.000253,0.000000,0.0,0.0,0.0,Art & photography,small,83.152376,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 15.0, 'weekly_regularity'..."
1612,1202223e,414,66,0.159420,2190695.0,33192.348485,0.575909,0.466350,0.830769,0.0,0.645003,0.024228,0.000000,0.0,0.0,0.0,Business services,small,82.706797,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 15.0, 'weekly_regularity'..."
19842,d67df067,298,28,0.093960,2246229.0,80222.464286,0.254975,0.189500,0.777778,0.0,0.645167,0.031432,0.000000,0.0,0.0,0.0,Business services,small,79.821134,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 7.0, 'weekly_regularity':..."
16301,b0cc04d0,407,63,0.154791,3581867.0,56855.031746,0.174395,0.068189,0.903226,0.0,0.992499,-0.001094,0.996687,0.0,0.0,0.0,Business services,small,78.021196,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 15.0, 'weekly_regularity'..."
11343,7af5c9b4,416,59,0.141827,759600.0,12874.576271,0.383230,0.561013,0.500000,0.0,0.947867,-0.005557,0.000000,0.0,0.0,0.0,Software,small,76.908055,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 14.75, 'weekly_regularity..."
5096,36fef85c,47,7,0.148936,117600.0,16800.000000,0.000000,0.000000,0.666667,0.0,1.000000,-0.059524,0.000000,0.0,0.0,0.0,Business services,small,76.754358,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 0.6854166666666667, 'week..."
22007,ee2a6901,91,12,0.131868,558000.0,46500.000000,0.000000,0.000000,0.818182,0.0,1.000000,-0.039560,0.000000,0.0,0.0,0.0,Construction,small,75.853386,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 2.275, 'weekly_regularity..."
5027,3664b731,408,66,0.161765,28950135.0,438638.409091,0.357792,0.383712,0.830769,0.0,0.120421,-0.001936,0.000000,0.0,0.0,0.0,Grocery & food stores,small,75.319272,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 15.0, 'weekly_regularity'..."
5445,3b022982,337,40,0.118694,970353.0,24258.825000,0.208013,0.257531,0.589744,0.0,0.954170,0.012459,0.000000,0.0,0.0,0.0,Personal services,small,75.152547,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 10.0, 'weekly_regularity'..."
8602,5cbaca44,142,24,0.169014,1796250.0,74843.750000,0.605410,0.599056,0.782609,0.0,0.746347,0.045441,0.000000,0.0,0.0,0.0,Business services,small,74.237313,Strong weekly cadence (many ~7-day gaps),"{'data_sufficiency': 6.0, 'weekly_regularity':..."


In [12]:
def product_reliance_row(r: pd.Series) -> str:
    parts = {
        'Subscriptions': r.get('frac_subscription', 0.0),
        'Checkout': r.get('frac_checkout', 0.0),
        'Payment Links': r.get('frac_payment_link', 0.0),
        'POS': r.get('frac_pos', 0.0),
    }
    parts = {k: v for k, v in parts.items() if v >= 0.05}
    if not parts:
        return 'Mixed/unclear (no dominant product)'
    return ', '.join([f"{k} {v:.0%}" for k, v in sorted(parts.items(), key=lambda kv: kv[1], reverse=True)])


def explanation_row(r: pd.Series) -> str:
    bd = r['score_breakdown']
    top = sorted(bd.items(), key=lambda kv: kv[1], reverse=True)[:3]

    reasons = []
    for k, v in top:
        if v < 3:
            continue
        if k == 'weekly_regularity':
            reasons.append('clear weekly cadence')
        elif k == 'monthly_regularity':
            reasons.append('clear monthly cadence')
        elif k in ['stability_daily', 'stability_weekly']:
            reasons.append('stable volumes over time')
        elif k == 'repeated_price_points':
            reasons.append('plan-like repeated amounts')
        elif k == 'subscription_opportunity':
            reasons.append('recurring behavior mostly outside Subscriptions (good migration opportunity)')
        elif k == 'data_sufficiency':
            reasons.append('enough history to be confident')
        elif k == 'volume_scale':
            reasons.append('meaningful payment scale')
        elif k == 'industry_fit':
            reasons.append('industry often fits subscriptions')

    product_mix = product_reliance_row(r)

    if not reasons:
        reasons = ['repeat activity over time']

    return f"{'; '.join(reasons)}. Current product mix: {product_mix}." 


ranked = mfeat.sort_values('candidate_score', ascending=False).copy()
ranked['product_reliance'] = ranked.apply(product_reliance_row, axis=1)
ranked['explanation'] = ranked.apply(explanation_row, axis=1)

cols = [
    'merchant', 'candidate_score', 'explanation', 'product_reliance', 'pattern_detected',
    'active_days', 'span_days', 'total_volume',
    'frac_subscription', 'frac_checkout', 'frac_payment_link', 'frac_pos',
    'cv_daily_active', 'cv_weekly', 'gap_share_7d', 'gap_share_30d', 'repeated_price_share_top3', 'trend_weekly_norm'
]
cols = [c for c in cols if c in ranked.columns]
ranked_out = ranked[cols]

ranked_out.head(10)

,merchant,candidate_score,explanation,product_reliance,pattern_detected,active_days,span_days,total_volume,frac_subscription,frac_checkout,frac_payment_link,frac_pos,cv_daily_active,cv_weekly,gap_share_7d,gap_share_30d,repeated_price_share_top3,trend_weekly_norm
3368,23ff5133,83.152376,enough history to be confident; stable volumes...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),76,418,5347126.0,0.000000,0.0,0.0,0.0,0.161254,0.377238,0.640000,0.0,0.849504,0.000253
1612,1202223e,82.706797,enough history to be confident; clear weekly c...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),66,414,2190695.0,0.000000,0.0,0.0,0.0,0.575909,0.466350,0.830769,0.0,0.645003,0.024228
19842,d67df067,79.821134,clear weekly cadence; recurring behavior mostl...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),28,298,2246229.0,0.000000,0.0,0.0,0.0,0.254975,0.189500,0.777778,0.0,0.645167,0.031432
16301,b0cc04d0,78.021196,enough history to be confident; clear weekly c...,Subscriptions 100%,Strong weekly cadence (many ~7-day gaps),63,407,3581867.0,0.996687,0.0,0.0,0.0,0.174395,0.068189,0.903226,0.0,0.992499,-0.001094
11343,7af5c9b4,76.908055,enough history to be confident; stable volumes...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),59,416,759600.0,0.000000,0.0,0.0,0.0,0.383230,0.561013,0.500000,0.0,0.947867,-0.005557
5096,36fef85c,76.754358,stable volumes over time; recurring behavior m...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),7,47,117600.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.666667,0.0,1.000000,-0.059524
22007,ee2a6901,75.853386,clear weekly cadence; stable volumes over time...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),12,91,558000.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.818182,0.0,1.000000,-0.039560
5027,3664b731,75.319272,enough history to be confident; clear weekly c...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),66,408,28950135.0,0.000000,0.0,0.0,0.0,0.357792,0.383712,0.830769,0.0,0.120421,-0.001936
5445,3b022982,75.152547,stable volumes over time; recurring behavior m...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),40,337,970353.0,0.000000,0.0,0.0,0.0,0.208013,0.257531,0.589744,0.0,0.954170,0.012459
8602,5cbaca44,74.237313,clear weekly cadence; recurring behavior mostl...,Mixed/unclear (no dominant product),Strong weekly cadence (many ~7-day gaps),24,142,1796250.0,0.000000,0.0,0.0,0.0,0.605410,0.599056,0.782609,0.0,0.746347,0.045441


In [13]:
# Deliverables: Top 10 strongest + Top 10 borderline

# Strongest candidates: top 10 overall
TOP_N = 10
strongest = ranked_out.head(TOP_N).copy()

# Borderline candidates: mid-high score but not top; also require enough data
borderline_pool = ranked[(ranked['active_days'] >= 20) & (ranked['span_days'] >= 45)].copy()
borderline_pool = borderline_pool.sort_values('candidate_score', ascending=False)
borderline = borderline_pool.iloc[TOP_N:TOP_N*2][cols].copy()

strongest, borderline

(       merchant  candidate_score                                        explanation                     product_reliance  \
 3368   23ff5133        83.152376  enough history to be confident; stable volumes...  Mixed/unclear (no dominant product)   
 1612   1202223e        82.706797  enough history to be confident; clear weekly c...  Mixed/unclear (no dominant product)   
 19842  d67df067        79.821134  clear weekly cadence; recurring behavior mostl...  Mixed/unclear (no dominant product)   
 16301  b0cc04d0        78.021196  enough history to be confident; clear weekly c...                   Subscriptions 100%   
 11343  7af5c9b4        76.908055  enough history to be confident; stable volumes...  Mixed/unclear (no dominant product)   
 5096   36fef85c        76.754358  stable volumes over time; recurring behavior m...  Mixed/unclear (no dominant product)   
 22007  ee2a6901        75.853386  clear weekly cadence; stable volumes over time...  Mixed/unclear (no dominant product)   


## Methodology (interpretable heuristic)

We identify **subscription adoption candidates** by looking for merchants with **repeat/recurring payment behavior** even if they currently use Subscriptions minimally.

**Feature groups**
- **Data sufficiency**: enough active days and time span to infer patterns.
- **Regularity (cadence)**: autocorrelation of daily volume at **7 days** (weekly cadence) and **30 days** (monthly cadence).
- **Stability / predictability**: coefficient of variation (CV) on daily and weekly totals (lower CV = more stable).
- **Repeated price points**: share of volume explained by the **top 3 repeated daily amounts** (after rounding), a proxy for plan-like pricing.
- **Subscription opportunity**: boost if recurring signals are strong but **subscription fraction is low** (good migration potential).
- **Scale sanity**: small boost for higher total volume so the list isn’t dominated by tiny but perfectly-regular merchants.
- **Industry fit (if available)**: small prior boost for industries commonly aligned with subscriptions.

**Candidate score**
A weighted 0–100 heuristic combining the above. Explanations are generated from the top scoring factors so results are business-friendly.

## Limitations / notes

- This uses **merchant-level daily totals** only; it cannot verify “same customer paying repeatedly” without customer/invoice-level data.
- Autocorrelation is sensitive to missing days; we reindex to a full daily calendar and treat missing days as 0, which is reasonable for cadence detection but can dampen signals for merchants with sporadic reporting.
- “Repeated price points” is approximate (rounded daily totals) and can miss cases where a merchant has multiple subscription tiers.
- The score is a **practical heuristic** (not a supervised model). You can tune weights to match business priorities (e.g., emphasize opportunity vs scale).

If you want, we can add plots per merchant (time series + weekly aggregates) for the top candidates to make the narrative even stronger.